In [1]:
import pandas as pd #load and work with tabular data.
from sklearn.model_selection import train_test_split #split dataset into training and testing parts.
import time #measure or use time functions.
import numpy as np #numerical operations with arrays.
from sklearn.preprocessing import StandardScaler #scale features to standard range.
from sklearn.feature_selection import SelectKBest #select best features using Chi‑Square test.
from sklearn.feature_selection import chi2 #select best features using Chi‑Square test.
from sklearn.feature_selection import RFE #recursive feature elimination (not used in this notebook).
from sklearn.linear_model import LogisticRegression #logistic regression model.
import pickle #save/load models.
import matplotlib.pyplot as plt #plotting graphs.

In [2]:
def selectkbest(indep_X,dep_Y,n): #picks top n features from data using Chi‑Square.
        test = SelectKBest(score_func=chi2, k=n) 
        fit1= test.fit(indep_X,dep_Y)
        selectk_features = fit1.transform(indep_X)
        return selectk_features
    
def split_scalar(indep_X,dep_Y): #splits data into train/test and scales features.
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test
    
 
def cm_prediction(classifier,X_test): #makes predictions, builds confusion matrix, calculates accuracy, and classification report.
     y_pred = classifier.predict(X_test)
        
        # Making the Confusion Matrix
     from sklearn.metrics import confusion_matrix
     cm = confusion_matrix(y_test, y_pred)
        
     from sklearn.metrics import accuracy_score 
     from sklearn.metrics import classification_report 
        #from sklearn.metrics import confusion_matrix
        #cm = confusion_matrix(y_test, y_pred)
        
     Accuracy=accuracy_score(y_test, y_pred )
        
     report=classification_report(y_test, y_pred)
     return  classifier,Accuracy,report,X_test,y_test,cm

def logistic(X_train,y_train,X_test):   #trains logistic regression and evaluates it.    
        # Fitting K-NN to the Training set
        from sklearn.linear_model import LogisticRegression
        classifier = LogisticRegression(random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm      
    
def svm_linear(X_train,y_train,X_test): #trains linear SVM and evaluates it.
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'linear', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
    
def svm_NL(X_train,y_train,X_test): #trains non‑linear (rbf kernel) SVM and evaluates it.
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'rbf', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
   
def Navie(X_train,y_train,X_test):  #trains Naive Bayes classifier and evaluates it.     
        # Fitting K-NN to the Training set
        from sklearn.naive_bayes import GaussianNB
        classifier = GaussianNB()
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm         
    
    
def knn(X_train,y_train,X_test): #trains K‑Nearest Neighbors classifier and evaluates it.
           
        # Fitting K-NN to the Training set
        from sklearn.neighbors import KNeighborsClassifier
        classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
def Decision(X_train,y_train,X_test): #trains Decision Tree classifier and evaluates it.
        
        # Fitting K-NN to the Training set
        from sklearn.tree import DecisionTreeClassifier
        classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm      


def random(X_train,y_train,X_test): #trains Random Forest classifier and evaluates it.
        
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestClassifier
        classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
    
def selectk_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf): #stores accuracy results of all models in a DataFrame.
    
    dataframe=pd.DataFrame(index=['ChiSquare'],columns=['Logistic','SVMl','SVMnl','KNN','Navie','Decision','Random'])
    for number,idex in enumerate(dataframe.index):      
        dataframe['Logistic'][idex]=acclog[number]       
        dataframe['SVMl'][idex]=accsvml[number]
        dataframe['SVMnl'][idex]=accsvmnl[number]
        dataframe['KNN'][idex]=accknn[number]
        dataframe['Navie'][idex]=accnav[number]
        dataframe['Decision'][idex]=accdes[number]
        dataframe['Random'][idex]=accrf[number]
    return dataframe

In [4]:
dataset1=pd.read_csv("prep.csv",index_col=None) #load dataset from CSV file.

df2=dataset1 #copy dataset.

df2 = pd.get_dummies(df2, drop_first=True) #convert categorical columns into numeric dummy variables.

indep_X=df2.drop('classification_yes', axis=1) #independent features (drop target column).
dep_Y=df2['classification_yes'] #dependent target column.

In [23]:
kbest=selectkbest(indep_X,dep_Y,7)     #select top 7 features.  

In [24]:
acclog=[] #Create empty lists (acclog, accsvml, etc.) → store accuracy values for each model.
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

X_train, X_test, y_train, y_test=split_scalar(kbest,dep_Y)   #split and scale data.
    
#Train each model (logistic, svm_linear, svm_NL, knn, Navie, Decision, random) → get accuracy and append to respective list.        
classifier,Accuracy,report,X_test,y_test,cm=logistic(X_train,y_train,X_test)
acclog.append(Accuracy)

classifier,Accuracy,report,X_test,y_test,cm=svm_linear(X_train,y_train,X_test)  
accsvml.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=svm_NL(X_train,y_train,X_test)  
accsvmnl.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=knn(X_train,y_train,X_test)  
accknn.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=Navie(X_train,y_train,X_test)  
accnav.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=Decision(X_train,y_train,X_test)  
accdes.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=random(X_train,y_train,X_test)  
accrf.append(Accuracy)
    
result=selectk_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf) #combine all accuracy results into one DataFrame.

C:\Users\user\.conda\envs\aiml\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
C:\Users\user\AppData\Local\Temp\ipykernel_17576\809484778.py:96: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.h

In [10]:
result #3 sets #shows a table with accuracy of Logistic, SVM (linear & non‑linear), KNN, Naive Bayes, Decision Tree, and Random Forest using selected features.

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
ChiSquare,0.82,0.82,0.82,0.85,0.8,0.84,0.83


In [14]:
result #4 sets

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
ChiSquare,0.85,0.82,0.83,0.86,0.79,0.89,0.89


In [19]:
result #5 sets

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
ChiSquare,0.94,0.94,0.95,0.89,0.83,0.96,0.95


In [22]:
result #6 sets

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
ChiSquare,0.95,0.96,0.96,0.93,0.89,0.97,0.97


In [25]:
result #7 sets

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
ChiSquare,0.97,0.97,0.97,0.97,0.91,0.98,0.97
